# InterScale Pipeline

In [ ]:
import scanpy as sc

import InterScale as interscale
from InterScale.config import load_config

# from graph_transformer_long_range_niches.config import load_config
# from graph_transformer_long_range_niches.pp import sliding_window

In [ ]:
CFG_PATH = "/home/icb/francesca.drummer/1-Projects/GT-long-range-niches/src/config_files/Legnini_23/legnini23_genes_sample_gnntrans.yaml"

## 0. Load and prepare data

We load a subset of the CosmX pancreas data containing T1D (type 1 diabetes) and ND (no diabetes) samples and the config file with the model specifications. 

In [ ]:
cfg = load_config(CFG_PATH)
cfg

In [ ]:
DATA_PATH = '/lustre/groups/ml01/projects/2024_spatial_long_range_GT_francesca.drummer/data/legnini23.h5ad'

In [ ]:
adata = sc.read_h5ad(DATA_PATH)
adata

If the samples are too large (more cells per sample than the transformers context length) then split the data into sliding windows. 

In [ ]:
sample_key = 'fov'

if adata.obs['fov'].mean() > cfg.transformer.context_length:
    SLIDING_WINDOW_KEY = 'sliding_window_square'
    sliding_window(
        adata,
        library_key = 'slide_fov',
        window_size = None,
        overlap = 0,
        max_n_cells=cfg.transformer.context_length,
        partial_window = 'merge',
        square = True,
        sliding_window_key = SLIDING_WINDOW_KEY,
        copy = False,
    ) 
    sample_key = SLIDING_WINDOW_KEY

Build a spatial neighborhood graph using `squidpy`.

In [ ]:
assert cfg.dataset.spatial_neigbors_kwargs.radius

## 1. Data setup

We need to specify:

- `prediction_task`: Prediction task can either be classification or regression
- `prediction_level`: Which level the predictions should be performed on: either (1) tissue label, e.i. condition (`graph`), (2) node label (`node`) for cell type or niche prediciton, or (3) GEX prediction.
- `prediction_obs`: Label in `adata.obs` to be predicted. Only required for classification tasks. 

In [6]:
prediction_task = 'classification'
prediction_level = 'graph'
prediction_obs = 'condition'

layer_key = 'log1p_norm'
sample_key = 'sample'
group_label = 'condition'

In [7]:
interscale.model.LocalModel._setup_anndata(adata = adata, layer_key = layer_key, sample_key = sample_key, prediction_obs = group_label)

/ictstr01/groups/ml01/workspace/francesca.drummer/mamba/envs/GT_long_range_env/lib/python3.11/site-packages/scvi/data/fields/_base_field.py:63: UserWarning: adata.layers[log1p_norm] does not contain unnormalized count data. Are you sure this is what you want?
  self.validate_field(adata)


Anndata setup with scvi-tools version 1.3.0.

     Summary Statistics     
┏━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Summary Stat Key ┃ Value ┃
┡━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ n_prediction_obs │   2   │
│   n_sample_key   │  17   │
│       n_x        │  88   │
└──────────────────┴───────┘

                    Data Registry                     
┏━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃  Registry Key  ┃        scvi-tools Location        ┃
┡━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ prediction_obs │ adata.obs['_scvi_prediction_obs'] │
│   sample_key   │   adata.obs['_scvi_sample_key']   │
│       x        │    adata.layers['log1p_norm']     │
└────────────────┴───────────────────────────────────┘

                prediction_obs State Registry                
┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃    Source Location     ┃ Categories ┃ scvi-tools Encoding ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ adata.obs['condition'] │    Ctrl    │          0          │
│                        │    SHH     │          1          │
└────────────────────────┴────────────┴─────────────────────┘

                 sample_key State Registry                 
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃   Source Location   ┃ Categories  ┃ scvi-tools Encoding ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ adata.obs['sample'] │ slide1_A2-1 │          0          │
│                     │ slide1_A2-2 │          1          │
│                     │ slide1_B2-1 │          2          │
│                     │ slide1_B2-2 │          3          │
│                     │ slide1_B2-3 │          4          │
│                     │ slide1_C2-1 │          5          │
│                     │ slide1_C2-2 │          6          │
│                     │ slide1_C2-3 │          7          │
│                     │ slide1_C2-5 │          8          │
│                     │ slide1_D2-2 │          9          │
│                     │ slide1_D2-3 │         10          │
│                     │ slide4_A2-1 │         11          │
│                     │ slide4_A2-2 │         12          │
│                     │ slide4_A2-3 │         13          │
│                     │ slide4_B2-1 │         14          │
│                     │ slide4_B2-2 │         15          │
│                     │ slide4_B2-3 │         16          │
└─────────────────────┴─────────────┴─────────────────────┘

In [8]:
adata

AnnData object with n_obs × n_vars = 43762 × 88
    obs: 'Cell', 'Area', 'x', 'y', 'sample', 'condition', 'organoid', 'obs_names', '_scvi_prediction_obs', '_scvi_sample_key'
    var: 'gene_ids', 'feature_types'
    uns: '_scvi_uuid', '_scvi_manager_uuid'
    obsm: 'spatial'
    layers: 'log1p_norm', 'norm_ftsqrt', 'raw'

## 2. Model setup

Define a LocalComponent and GlobalComponent.

In [9]:
GCN = interscale.model.LocalModel(
    adata,
    cfg,
    local_component_name = 'GCN'
)

TypeError: Can't instantiate abstract class LocalModel with abstract method train

In [ ]:
assert cfg.dataset.prediction_task.isin(['node_classification', 'sample_classification', 'node_regression'])
assert cfg.dataset.prediction_task.isin(adata.obs_names)

## 3. Training

In [ ]:
model.train()

## 4. Evaluation

Evaluation can either be run on the entire AnnData that the model was set up with or subsets of AnnData defined by sample_id from `adata.obs[sample_key]`.

In [ ]:
embedding = model.local_component.get_embedding(sample_id = [],
                                save_results = False)